        # Análisis de comportamiento de sistemas bajo diferentes condiciones

        **Modelación y Simulación Computacional** · Maestría en Ingeniería ·
        Universidad de Sucre · periodo 2026-2

        **Unidad 3.** Simulación de sistemas y análisis de escenarios ·
        **Subtema del plan 3.2**

        Autor, Prof. Daniel Otero Meza, Ing., Ph.D.

        <!-- ENLACE_COLAB -->
        [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/<usuario>/<repositorio>/blob/main/03_cuadernos/Unidad3/U3_02_comportamiento_bajo_condiciones.ipynb)

        Una corrida aislada no dice nada sobre el comportamiento de un sistema.
Este cuaderno cambia de manera deliberada las condiciones de operación,
las tolerancias, el método de integración, la malla y el paso de tiempo,
y observa qué cambia en la respuesta y qué cambia solo en el costo. La
distinción entre ambas cosas es lo que separa un resultado defendible de
una gráfica bonita.

        ## Objetivos de aprendizaje

        Al terminar este cuaderno el estudiante debe ser capaz de

        1. Comparar la respuesta de un mismo modelo dinámico bajo tres condiciones de operación y verificar la relación de conservación que comparten.
2. Distinguir la tolerancia del error, midiendo el error global frente a una referencia de alta precisión al variar la tolerancia solicitada.
3. Calcular la razón de rigidez de la Definición 3.3 a partir de los autovalores del jacobiano y anticipar el paso máximo de un método explícito.
4. Diagnosticar la rigidez comparando el conteo de pasos y de evaluaciones de un método explícito y de uno implícito.
5. Verificar el orden de convergencia espacial de un método de líneas y el límite de estabilidad de su versión explícita.

## Puesta a punto

La primera celda detecta el entorno e instala solo lo que falte, de modo que
el cuaderno abre igual en Google Colab y en JupyterLab. La segunda fija la
semilla del curso, la paleta del libro y las funciones auxiliares. La semilla
vale 20262 y ningún resultado depende de una ejecución concreta.

In [ ]:
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict) -> None:
    """Instala solo los paquetes que no estén disponibles."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("entorno listo, Colab =", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

COLORES = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({"figure.figsize": (9.0, 4.4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10, "legend.frameon": False})

trapecio = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

pd.set_option("display.width", 110)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def carpeta_datos() -> Path:
    """Ubica la carpeta datos sin usar rutas absolutas.

    Busca hacia arriba desde el directorio de trabajo, de modo que funcione
    tanto en el repositorio como en una sesión de Colab donde el cuaderno se
    abre suelto. Si no la encuentra, la crea junto al cuaderno.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        if (candidata / "datos").is_dir():
            return candidata / "datos"
    destino = base / "datos"
    destino.mkdir(exist_ok=True)
    return destino


def leer_datos(nombre: str, respaldo) -> pd.DataFrame:
    """Lee un archivo de datos y lo reconstruye si no está disponible.

    El argumento respaldo es una función sin argumentos que devuelve el
    mismo cuadro de datos, construido con las cifras publicadas en el libro.
    Así el cuaderno nunca depende de una descarga.
    """
    base = Path.cwd()
    for candidata in (base, *list(base.parents)[:3]):
        ruta = candidata / "datos" / nombre
        if ruta.exists():
            return pd.read_csv(ruta)
    tabla = respaldo()
    tabla.to_csv(carpeta_datos() / nombre, index=False)
    return tabla


def comparar(etiqueta: str, calculado: float, libro: float,
             tol: float, unidad: str = "") -> bool:
    """Imprime y verifica un valor calculado frente al que publica el libro."""
    dif = abs(calculado - libro)
    ok = dif <= tol
    marca = "coincide" if ok else "NO coincide"
    print(f"{etiqueta:<46s} calculado {calculado:>14.6g} {unidad:<12s}"
          f" libro {libro:>12.6g}   {marca}")
    return ok


print("semilla del curso", SEMILLA)

In [ ]:
def respaldo_valores_libro() -> pd.DataFrame:
    """Cifras publicadas en el capítulo 3, transcritas del libro."""
    filas = [
    ("colebrook_velocidad", 1.6977, "m/s", "Ejemplo 3.1"),
    ("colebrook_reynolds", 507267.0, "adimensional", "Ejemplo 3.1"),
    ("colebrook_rugosidad_relativa", 0.0008667, "adimensional", "Ejemplo 3.1"),
    ("colebrook_factor_friccion", 0.0196228, "adimensional", "Ejemplo 3.1"),
    ("colebrook_perdida_carga", 8.167, "m", "Ejemplo 3.1"),
    ("colebrook_swamee_jain", 0.019742, "adimensional", "Ejemplo 3.1"),
    ("colebrook_orden_newton", 2.0, "adimensional", "Ejemplo 3.1"),
    ("lagunas_perfil_1", 142.42, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_2", 83.4, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_3", 41.5, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_4", 17.65, "mg/L", "seccion 3.1.2"),
    ("lagunas_perfil_5", 7.33, "mg/L", "seccion 3.1.2"),
    ("lagunas_remocion", 97.07, "por ciento", "seccion 3.1.2"),
    ("lagunas_retencion", 8.33, "d", "seccion 3.1.2"),
    ("lagunas_carga_afluente", 300000.0, "mg/d", "seccion 3.1.2"),
    ("lagunas_carga_efluente", 8792.84, "mg/d", "seccion 3.1.2"),
    ("lagunas_consumo", 291207.16, "mg/d", "seccion 3.1.2"),
    ("fermentador_tiempo_25C", 17.0604614, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_30C", 10.9186953, "h", "Ejemplo 3.2"),
    ("fermentador_tiempo_35C", 7.5824273, "h", "Ejemplo 3.2"),
    ("fermentador_invariante", 12.5, "g/L", "Ejemplo 3.2"),
    ("fermentador_mumax_25C", 0.192, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_30C", 0.3, "1/h", "Ejemplo 3.2"),
    ("fermentador_mumax_35C", 0.432, "1/h", "Ejemplo 3.2"),
    ("fermentador_evaluaciones", 584.0, "evaluaciones", "Ejemplo 3.2"),
    ("tolerancia_tiempo_rtol3", 10.9028, "h", "seccion 3.2.1"),
    ("tolerancia_error_rtol3", 0.00146, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol3", 44.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol6", 1.85e-07, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol6", 194.0, "evaluaciones", "seccion 3.2.1"),
    ("tolerancia_error_rtol9", 2.45e-10, "adimensional", "seccion 3.2.1"),
    ("tolerancia_evaluaciones_rtol9", 584.0, "evaluaciones", "seccion 3.2.1"),
    ("circuito_autovalor_rapido", -1005.0002, "1/s", "Ejemplo 3.3"),
    ("circuito_autovalor_lento", -0.0497512, "1/s", "Ejemplo 3.3"),
    ("circuito_razon_rigidez", 20200.0, "adimensional", "Ejemplo 3.3"),
    ("circuito_pasos_rk45", 30376.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_rk45", 212576.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_pasos_bdf", 144.0, "pasos", "Ejemplo 3.3"),
    ("circuito_evaluaciones_bdf", 292.0, "evaluaciones", "Ejemplo 3.3"),
    ("circuito_paso_medio_rk45", 0.003292, "s", "Ejemplo 3.3"),
    ("circuito_tau_rapida", 0.000995, "s", "Ejemplo 3.3"),
    ("circuito_tau_lenta", 20.1, "s", "Ejemplo 3.3"),
    ("circuito_error_rk45", 9.1e-07, "adimensional", "Ejemplo 3.3"),
    ("circuito_error_bdf", 1.1e-06, "adimensional", "Ejemplo 3.3"),
    ("circuito_producto_h_lambda", 3.31, "adimensional", "Ejemplo 3.3"),
    ("rio_peclet_celda", 0.583, "adimensional", "Ejemplo 3.4"),
    ("rio_pico_analitico", 1.8655, "mg/L", "Ejemplo 3.4"),
    ("rio_abscisa_pico", 2460.0, "m", "Ejemplo 3.4"),
    ("rio_error_maximo", 7.18e-06, "kg/m3", "Ejemplo 3.4"),
    ("rio_masa_remanente", 24.740935, "kg", "Ejemplo 3.4"),
    ("rio_orden_observado", 2.0, "adimensional", "Ejemplo 3.4"),
    ("rio_paso_difusion", 16.67, "s", "seccion 3.3.2"),
    ("rio_paso_adveccion", 57.14, "s", "seccion 3.3.2"),
    ("rio_error_explicito_d045", 6.3e-05, "kg/m3", "Ejemplo 3.4"),
    ("riego_frontera_bruto_p045_d45", 393.8, "mm", "Ejemplo 3.5"),
    ("riego_frontera_bruto_p085_d65", 243.8, "mm", "Ejemplo 3.5"),
    ("riego_deficit_p085_d65", 21.5, "por ciento", "Ejemplo 3.5"),
    ("riego_no_dominadas_clima_normal", 5.0, "alternativas", "Ejemplo 3.5"),
    ("biogas_desviacion_replicas_mc", 0.933, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion_replicas_lhs", 0.112, "kW h/d", "Ejemplo 3.6"),
    ("biogas_reduccion_varianza", 69.0, "veces", "Ejemplo 3.6"),
    ("riego_agua_aprovechable", 126.0, "mm", "Ejemplo 3.5"),
    ("biogas_media", 92.34, "kW h/d", "Ejemplo 3.6"),
    ("biogas_desviacion", 21.22, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p05", 61.71, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p50", 90.05, "kW h/d", "Ejemplo 3.6"),
    ("biogas_p95", 130.62, "kW h/d", "Ejemplo 3.6"),
    ("biogas_excedencia_110", 0.1945, "adimensional", "Ejemplo 3.6"),
    ("biogas_error_estandar", 0.15, "kW h/d", "Ejemplo 3.6"),
    ("biogas_valores_centrales", 91.53, "kW h/d", "Ejemplo 3.6"),
    ("lcoe_crf", 0.101806, "1/a", "Ejemplo 3.7"),
    ("lcoe_factor_degradacion", 0.931205, "adimensional", "Ejemplo 3.7"),
    ("lcoe_produccion_especifica", 1325.6, "kW h/(kW a)", "Ejemplo 3.7"),
    ("lcoe_nominal", 0.083523, "USD/(kW h)", "Ejemplo 3.7"),
    ("lcoe_elasticidad_inversion", 0.87355, "adimensional", "Ejemplo 3.7"),
    ("lcoe_elasticidad_tasa", 0.637, "adimensional", "Ejemplo 3.7"),
    ("lcoe_amplitud_tasa", 35.8, "por ciento", "Ejemplo 3.7"),
    ("lcoe_amplitud_irradiacion", 16.1, "por ciento", "Ejemplo 3.7"),
    ("ishigami_s1", 0.3138, "adimensional", "seccion 3.6.2"),
    ("ishigami_s2", 0.4423, "adimensional", "seccion 3.6.2"),
    ("ishigami_s3", -0.0001, "adimensional", "seccion 3.6.2"),
    ("ishigami_st3", 0.2436, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_s_B0", 0.616, "adimensional", "seccion 3.6.2"),
    ("sobol_biogas_suma_primer_orden", 0.985, "adimensional", "seccion 3.6.2"),
    ("morris_evaluaciones", 210.0, "evaluaciones", "seccion 3.6.2"),
    ]
    return pd.DataFrame(filas, columns=["clave", "valor", "unidad", "referencia"])


LIBRO = leer_datos("valores_libro_cap3.csv",
                   respaldo_valores_libro).set_index("clave")["valor"]
print(f"cifras del libro disponibles, {LIBRO.size} registros")

## 1. El mismo modelo bajo tres condiciones de operación

El Ejemplo 3.2 del libro fermenta jugo de caña a 25, 30 y 35 grados
Celsius. La cinética es de Monod y la velocidad específica máxima depende
de la temperatura según el modelo de la raíz cuadrada,
\(\mu_{\max}(T) = \mu_{\mathrm{ref}}\left[(T-T_{\min})/(T_{\mathrm{ref}}-T_{\min})\right]^{2}\),
con \(\mu_{\mathrm{ref}} = 0.30\) por hora a 30 grados y
\(T_{\min} = 5\) grados. La Figura 3.5 muestra las tres trayectorias
con el instante de agotamiento marcado sobre cada curva.

In [ ]:
from scipy.integrate import quad, solve_ivp

KS_MONOD, RENDIMIENTO = 1.2, 0.08        # g/L y adimensional
X0, S0_LOTE = 0.50, 150.0                # g/L
T_MIN, T_REF, MU_REF = 5.0, 30.0, 0.30   # grados Celsius y 1/h


def fermentador(t, y, mu_max):
    """Campo del fermentador en lote, con X y S en g/L."""
    X, S = y
    mu = mu_max * S / (KS_MONOD + S)
    return [mu * X, -mu * X / RENDIMIENTO]


def agotamiento(t, y, mu_max):
    """Evento terminal que detecta el descenso del sustrato a 1 g/L."""
    return y[1] - 1.0


agotamiento.direction = -1
agotamiento.terminal = True

### Ejercicio 1

Escriba la velocidad específica máxima en función de la temperatura según
el modelo de la raíz cuadrada. La celda de partida devuelve un valor fijo
de 0.30 por hora, con lo cual las tres temperaturas producen la misma
curva y el efecto de la condición de operación desaparece.

In [ ]:
# COMPLETE: mu_max(T) = MU_REF*((T - T_MIN)/(T_REF - T_MIN))**2
REVISAR_MU = False


def mu_maxima(T: float) -> float:
    """Velocidad específica máxima en 1/h para la temperatura en grados."""
    return MU_REF          # marcador de posición, no depende de T

In [ ]:
TEMPERATURAS = (25.0, 30.0, 35.0)
corridas = {}
for T in TEMPERATURAS:
    sol = solve_ivp(fermentador, (0.0, 40.0), [X0, S0_LOTE],
                    args=(mu_maxima(T),), method="RK45", rtol=1e-9,
                    atol=1e-11, dense_output=True, events=agotamiento)
    corridas[T] = sol

tabla = pd.DataFrame(
    [(T, mu_maxima(T), float(corridas[T].t_events[0][0]), corridas[T].nfev)
     for T in TEMPERATURAS],
    columns=["T (grados C)", "mu_max (1/h)", "agotamiento (h)",
             "evaluaciones"])
print(tabla.to_string(index=False))

ok = []
for T, clave in zip(TEMPERATURAS, ("25C", "30C", "35C")):
    ok.append(comparar(f"velocidad máxima a {T:.0f} grados", mu_maxima(T),
                       LIBRO[f"fermentador_mumax_{clave}"], 5e-4, "1/h"))
    ok.append(comparar(f"agotamiento a {T:.0f} grados",
                       float(corridas[T].t_events[0][0]),
                       LIBRO[f"fermentador_tiempo_{clave}"], 1e-4, "h"))

if REVISAR_MU:
    assert all(ok), "los tiempos no coinciden con el Ejemplo 3.2"
    assert all(corridas[T].nfev == 584 for T in TEMPERATURAS), \
        "las tres corridas deben costar 584 evaluaciones del campo"
    print("los tres tiempos y el costo coinciden con el Ejemplo 3.2")
else:
    print("complete la celda anterior y ponga REVISAR_MU = True")

Las tres trayectorias comparten el estado final de biomasa, consecuencia
de la relación de conservación \(X + Y S = 12.5\) g/L que se deduce del
propio modelo, y difieren solo en la escala de tiempo. Esa coincidencia
es una verificación exacta y gratuita. Un aumento de diez grados reduce
el tiempo de proceso a menos de la mitad, pero el modelo no contempla la
pérdida de viabilidad celular por encima de 35 grados, de modo que
extrapolarlo a 40 produciría un número plausible y falso.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
for T, color in zip(TEMPERATURAS, (COLORES["azul"], COLORES["verde"],
                                   COLORES["rojo"])):
    sol = corridas[T]
    t = np.linspace(0.0, float(sol.t[-1]), 600)
    X, S = sol.sol(t)
    ax1.plot(t, S, color=color, label=f"T = {T:.0f} grados C")
    ax2.plot(t, X, color=color)
    te = float(sol.t_events[0][0])
    ax1.plot([te], [1.0], "o", color=color, ms=5, mfc="white")
    ax2.axvline(te, color=color, lw=0.7, ls=":")
ax1.set_xlabel("tiempo t (h)")
ax1.set_ylabel("sustrato S (g/L)")
ax1.set_xlim(0, 20)
ax1.legend()
ax2.axhline(12.5, color=COLORES["gris"], ls="--", lw=0.9)
ax2.text(0.6, 11.4, "X + Y S = 12.5 g/L", color=COLORES["gris"], fontsize=9)
ax2.set_xlabel("tiempo t (h)")
ax2.set_ylabel("biomasa X (g/L)")
ax2.set_xlim(0, 20)
fig.suptitle("Fermentación en lote a tres temperaturas de operación")
plt.tight_layout()
plt.show()

## 2. La tolerancia no es el error

La Ecuación 3.9 del libro construye la cota del error local con una
tolerancia relativa y una absoluta. Conviene insistir en un punto que se
malinterpreta con frecuencia. Las tolerancias controlan el error local
estimado de cada paso, no el error al final del horizonte. La única
manera honesta de conocer el error global consiste en repetir la corrida
con tolerancias más estrictas hasta que la respuesta deje de cambiar, o
en compararla con una referencia independiente.

Aquí la referencia es la cuadratura sobre el invariante, que entrega el
instante de agotamiento con catorce cifras.

In [ ]:
CONSTANTE = X0 + RENDIMIENTO * S0_LOTE     # g/L


def tiempo_por_cuadratura(mu_max: float, s_final: float = 1.0) -> float:
    """Referencia de alta precisión obtenida del invariante del modelo."""
    integrando = lambda s: -(RENDIMIENTO * (KS_MONOD + s)
                             / (mu_max * s * (CONSTANTE - RENDIMIENTO * s)))
    valor, _ = quad(integrando, S0_LOTE, s_final, epsabs=1e-13,
                    epsrel=1e-13, limit=400)
    return float(valor)


referencia = tiempo_por_cuadratura(mu_maxima(30.0))
print(f"referencia por cuadratura a 30 grados   {referencia:.10f} h")

### Ejercicio 2

Complete el estudio de tolerancias. Para cada tolerancia relativa de la
lista, integre el fermentador a 30 grados con tolerancia absoluta mil
veces menor que la relativa, recoja el instante de agotamiento, el número
de evaluaciones del campo y el error relativo frente a la referencia. La
celda de partida deja fija la tolerancia en 10⁻³ y no varía nada.

In [ ]:
# COMPLETE: use rtol de la lista y atol = rtol*1e-3 en cada corrida.
REVISAR_TOL = False


def estudio_tolerancias(rtols, mu_max, referencia):
    """Instante, costo y error relativo para cada tolerancia pedida."""
    filas = []
    for rtol in rtols:
        sol = solve_ivp(fermentador, (0.0, 40.0), [X0, S0_LOTE],
                        args=(mu_max,), method="RK45",
                        rtol=1e-3, atol=1e-6,   # falta usar rtol
                        dense_output=True, events=agotamiento)
        te = float(sol.t_events[0][0])
        filas.append((rtol, te, sol.nfev, abs(te - referencia) / referencia))
    return pd.DataFrame(filas, columns=["rtol", "agotamiento (h)",
                                        "evaluaciones", "error relativo"])

In [ ]:
estudio = estudio_tolerancias([1e-3, 1e-6, 1e-9], mu_maxima(30.0), referencia)
print(estudio.to_string(index=False,
                        formatters={"rtol": "{:.0e}".format,
                                    "error relativo": "{:.3e}".format}))

ok = [comparar("instante con rtol 1e-3", estudio.loc[0, "agotamiento (h)"],
               LIBRO["tolerancia_tiempo_rtol3"], 5e-4, "h"),
      comparar("error relativo con rtol 1e-3", estudio.loc[0, "error relativo"],
               LIBRO["tolerancia_error_rtol3"], 5e-6, ""),
      comparar("error relativo con rtol 1e-6", estudio.loc[1, "error relativo"],
               LIBRO["tolerancia_error_rtol6"], 5e-9, ""),
      comparar("evaluaciones con rtol 1e-3", estudio.loc[0, "evaluaciones"],
               LIBRO["tolerancia_evaluaciones_rtol3"], 0, ""),
      comparar("evaluaciones con rtol 1e-6", estudio.loc[1, "evaluaciones"],
               LIBRO["tolerancia_evaluaciones_rtol6"], 0, ""),
      comparar("evaluaciones con rtol 1e-9", estudio.loc[2, "evaluaciones"],
               LIBRO["tolerancia_evaluaciones_rtol9"], 0, "")]
print()
print("error relativo con rtol 1e-9, calculado "
      f"{estudio.loc[2, 'error relativo']:.3e}, "
      f"libro {LIBRO['tolerancia_error_rtol9']:.3e}")
print("la diferencia está en la tercera cifra del propio estimado de error, "
      "que depende de la referencia con la cual se compara")

crecimiento = (estudio.loc[2, "evaluaciones"] / estudio.loc[0, "evaluaciones"])
print(f"\ncrecimiento del costo entre rtol 1e-3 y 1e-9   {crecimiento:.1f} veces")
print(f"crecimiento que anuncia la ley rtol**(-1/5)    {(1e6)**0.2:.1f} veces")

if REVISAR_TOL:
    assert all(ok), "el estudio de tolerancias no reproduce la sección 3.2.1"
    assert estudio.loc[0, "error relativo"] > 1e-3, \
        "con rtol 1e-3 el error global supera la tolerancia pedida"
    print("\nel error global excede la tolerancia solicitada, tal como advierte el libro")
else:
    print("complete la celda anterior y ponga REVISAR_TOL = True")

## 3. Estabilidad absoluta y rigidez

Hasta aquí el paso lo ha fijado la exactitud. Existe una segunda
restricción que no se manifiesta como pérdida de precisión sino como una
explosión numérica. Aplicado a la ecuación de prueba \(y' = \lambda y\),
un método de un paso produce la recurrencia \(y_{n+1} = R(z)\,y_n\) con
\(z = h\lambda\), y el conjunto donde \(|R(z)| \le 1\) es su región de
estabilidad absoluta. La Figura 3.6 del libro compara tres regiones y la
celda siguiente las reconstruye.

In [ ]:
rez = np.linspace(-4.0, 2.0, 500)
imz = np.linspace(-3.5, 3.5, 500)
Z = rez[None, :] + 1j * imz[:, None]

R_euler = 1 + Z
R_rk4 = 1 + Z + Z**2 / 2 + Z**3 / 6 + Z**4 / 24
R_implicito = 1 / (1 - Z)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.contourf(rez, imz, np.abs(R_implicito), levels=[0.0, 1.0],
            colors=[COLORES["verde"]], alpha=0.16)
ax.contour(rez, imz, np.abs(R_euler), levels=[1.0], colors=[COLORES["azul"]])
ax.contour(rez, imz, np.abs(R_rk4), levels=[1.0], colors=[COLORES["naranja"]])
ax.contour(rez, imz, np.abs(R_implicito), levels=[1.0], colors=[COLORES["verde"]])
ax.axhline(0, color=COLORES["gris"], lw=0.6)
ax.axvline(0, color=COLORES["gris"], lw=0.6)
ax.set_xlabel("parte real de h por lambda")
ax.set_ylabel("parte imaginaria de h por lambda")
ax.set_title("Regiones de estabilidad absoluta")
ax.plot([], [], color=COLORES["azul"], label="Euler explícito")
ax.plot([], [], color=COLORES["naranja"], label="Runge-Kutta de orden cuatro")
ax.plot([], [], color=COLORES["verde"], label="Euler implícito, A estable")
ax.legend(loc="upper left", fontsize=8.5)
ax.set_aspect("equal")
plt.show()

El Ejemplo 3.3 alimenta con 5 V, a través de una resistencia de 1 kiloohm,
un nodo con capacitancia a tierra de 1 microfaradio, y conecta ese nodo
mediante 200 kiloohm a un segundo nodo con 100 microfaradios. Las leyes de
Kirchhoff conducen a un sistema lineal cuyos autovalores difieren en
cuatro órdenes de magnitud.

### Ejercicio 3

Complete la razón de rigidez de la Definición 3.3 y el paso máximo que la
estabilidad permite a Euler explícito, que vale \(2/|\lambda_{\max}|\).

In [ ]:
R1, C1, R2, C2, VS = 1.0e3, 1.0e-6, 2.0e5, 1.0e-4, 5.0
MATRIZ = np.array([[-(1 / R1 + 1 / R2) / C1, (1 / R2) / C1],
                   [(1 / R2) / C2, -(1 / R2) / C2]])
FUENTE = np.array([VS / (R1 * C1), 0.0])
autovalores = np.linalg.eigvals(MATRIZ)
constantes = 1 / np.abs(autovalores)

comparar("autovalor rápido", np.min(autovalores),
         LIBRO["circuito_autovalor_rapido"], 5e-4, "1/s")
comparar("autovalor lento", np.max(autovalores),
         LIBRO["circuito_autovalor_lento"], 5e-7, "1/s")
comparar("constante de tiempo rápida", np.min(constantes),
         LIBRO["circuito_tau_rapida"], 5e-7, "s")
comparar("constante de tiempo lenta", np.max(constantes),
         LIBRO["circuito_tau_lenta"], 5e-3, "s")

In [ ]:
# COMPLETE: razon_rigidez = max|lambda| / min|lambda|
#           paso_maximo   = 2 / max|lambda|
REVISAR_RIGIDEZ = False


def razon_rigidez(autovalores) -> float:
    """Razón entre el autovalor de mayor y el de menor magnitud."""
    return 1.0            # marcador de posición


def paso_maximo_euler(autovalores) -> float:
    """Paso máximo admisible para Euler explícito, en segundos."""
    return 1.0            # marcador de posición

In [ ]:
rigidez = razon_rigidez(autovalores)
h_max = paso_maximo_euler(autovalores)
comparar("razón de rigidez", rigidez, LIBRO["circuito_razon_rigidez"], 2.0, "")
print(f"paso máximo de Euler explícito                 {h_max * 1e3:.3f} ms")
print(f"pasos necesarios para 100 s con ese paso       {100 / h_max:,.0f}")

if REVISAR_RIGIDEZ:
    assert abs(rigidez - LIBRO["circuito_razon_rigidez"]) < 2.0, \
        "la razón de rigidez debe valer cerca de 20200"
    assert abs(h_max - 2 / np.abs(autovalores).max()) < 1e-12
    print("la razón de rigidez coincide con el Ejemplo 3.3")
else:
    print("complete la celda anterior y ponga REVISAR_RIGIDEZ = True")

## 4. Diagnóstico de la rigidez por conteo de evaluaciones

El diagnóstico y el remedio se aplican en el mismo gesto. El Listado 3.4
del libro integra el problema con un método explícito y con uno implícito
e imprime pasos y evaluaciones. Un cociente de dos o tres órdenes de
magnitud entre ambos conteos es la firma inequívoca de la rigidez. El
sistema es lineal y admite solución exacta partiendo del reposo, que
sirve de referencia.

In [ ]:
from scipy.linalg import expm

HORIZONTE = 100.0
campo_rc = lambda t, v: MATRIZ @ v + FUENTE
jac_rc = lambda t, v: MATRIZ
exacta_rc = (np.linalg.inv(MATRIZ)
             @ (expm(MATRIZ * HORIZONTE) - np.eye(2)) @ FUENTE)

corridas_rc = {}
for metodo in ("RK45", "BDF"):
    extra = {"jac": jac_rc} if metodo == "BDF" else {}
    corridas_rc[metodo] = solve_ivp(campo_rc, (0.0, HORIZONTE), np.zeros(2),
                                    method=metodo, rtol=1e-6, atol=1e-9,
                                    **extra)

filas = []
for metodo, sol in corridas_rc.items():
    error = float(np.max(np.abs(sol.y[:, -1] - exacta_rc) / np.abs(exacta_rc)))
    filas.append((metodo, sol.t.size - 1, sol.nfev, error,
                  HORIZONTE / (sol.t.size - 1)))
comparacion = pd.DataFrame(filas, columns=["método", "pasos", "evaluaciones",
                                           "error relativo", "paso medio (s)"])
print(comparacion.to_string(index=False,
                            formatters={"error relativo": "{:.2e}".format}))
print(f"\nrelación de costo entre RK45 y BDF   "
      f"{corridas_rc['RK45'].nfev / corridas_rc['BDF'].nfev:.0f} a uno")

In [ ]:
ok = [comparar("pasos de RK45", comparacion.loc[0, "pasos"],
               LIBRO["circuito_pasos_rk45"], 0, ""),
      comparar("evaluaciones de RK45", comparacion.loc[0, "evaluaciones"],
               LIBRO["circuito_evaluaciones_rk45"], 0, ""),
      comparar("pasos de BDF", comparacion.loc[1, "pasos"],
               LIBRO["circuito_pasos_bdf"], 0, ""),
      comparar("evaluaciones de BDF", comparacion.loc[1, "evaluaciones"],
               LIBRO["circuito_evaluaciones_bdf"], 0, ""),
      comparar("error relativo de RK45", comparacion.loc[0, "error relativo"],
               LIBRO["circuito_error_rk45"], 5e-8, ""),
      comparar("error relativo de BDF", comparacion.loc[1, "error relativo"],
               LIBRO["circuito_error_bdf"], 5e-8, ""),
      comparar("paso medio de RK45", comparacion.loc[0, "paso medio (s)"],
               LIBRO["circuito_paso_medio_rk45"], 5e-7, "s")]
producto = comparacion.loc[0, "paso medio (s)"] * np.abs(autovalores).max()
ok.append(comparar("producto h por lambda máximo", producto,
                   LIBRO["circuito_producto_h_lambda"], 5e-3, ""))
assert all(ok), "los conteos no reproducen el Ejemplo 3.3"
print("el paso lo fija la estabilidad y no la exactitud")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
t_fino = np.concatenate([np.logspace(-5, np.log10(HORIZONTE), 900), [HORIZONTE]])
exacto = np.array([np.linalg.inv(MATRIZ) @ (expm(MATRIZ * ti) - np.eye(2)) @ FUENTE
                   for ti in t_fino])
ax1.semilogx(t_fino, exacto[:, 0], color=COLORES["gris"], label="v1 exacta")
ax1.semilogx(t_fino, exacto[:, 1], color=COLORES["azul"], label="v2 exacta")
ax1.semilogx(corridas_rc["BDF"].t[1:], corridas_rc["BDF"].y[1][1:], "o",
             color=COLORES["rojo"], ms=3.0, mfc="none", label="pasos de BDF")
for tau, etiqueta in zip(np.sort(constantes), ("tau rápida", "tau lenta")):
    ax1.axvline(tau, color=COLORES["verde"], lw=0.8, ls=":")
ax1.set_xlabel("tiempo t (s)")
ax1.set_ylabel("tensión (V)")
ax1.legend(loc="upper left", fontsize=8.5)

for metodo, color in (("RK45", COLORES["azul"]), ("BDF", COLORES["rojo"])):
    sol = corridas_rc[metodo]
    ax2.semilogy(sol.t[1:], np.diff(sol.t), color=color, lw=1.0,
                 label=f"{metodo}, {sol.nfev} evaluaciones")
ax2.axhline(3.30657 / np.abs(autovalores).max(), color=COLORES["verde"],
            lw=1.0, ls="--")
ax2.set_xlabel("tiempo t (s)")
ax2.set_ylabel("paso aceptado h (s)")
ax2.legend(loc="upper left", fontsize=8.5)
fig.suptitle("Circuito con dos constantes de tiempo separadas")
plt.tight_layout()
plt.show()

### Ejercicio 4

El problema 3-28 del capítulo pide una función que compare tres
integradores sobre un mismo problema y devuelva pasos, evaluaciones y
error frente a una referencia. Complete el cuerpo del bucle. La celda de
partida devuelve ceros, con lo cual la comparación no informa nada.

In [ ]:
# COMPLETE: para cada método, integre con solve_ivp y recoja
#   pasos = sol.t.size - 1, evaluaciones = sol.nfev y el error
#   relativo máximo del estado final frente a la referencia.
REVISAR_COMPARA = False


def comparar_integradores(campo, jac, intervalo, y0, referencia,
                          metodos=("RK45", "BDF", "LSODA"),
                          rtol=1e-6, atol=1e-9) -> pd.DataFrame:
    """Pasos, evaluaciones y error de cada integrador."""
    filas = []
    for metodo in metodos:
        filas.append((metodo, 0, 0, 0.0))
    return pd.DataFrame(filas, columns=["método", "pasos",
                                        "evaluaciones", "error relativo"])

In [ ]:
tres = comparar_integradores(campo_rc, jac_rc, (0.0, HORIZONTE), np.zeros(2),
                             exacta_rc)
print(tres.to_string(index=False,
                     formatters={"error relativo": "{:.2e}".format}))

if REVISAR_COMPARA:
    assert int(tres.loc[0, "evaluaciones"]) == 212576, \
        "RK45 debe gastar 212576 evaluaciones"
    assert int(tres.loc[1, "evaluaciones"]) == 292, \
        "BDF con jacobiano debe gastar 292 evaluaciones"
    assert tres.loc[2, "evaluaciones"] < tres.loc[0, "evaluaciones"] / 100, \
        "LSODA debe detectar la rigidez y conmutar a la familia implícita"
    print("\nLSODA conmuta de manera automática y queda del lado implícito")
else:
    print("complete la celda anterior y ponga REVISAR_COMPARA = True")

## 5. El efecto de la malla y del paso en un sistema distribuido

El Algoritmo 3.2 del libro enumera tres controles que separan una
implementación defendible de una que solo produce gráficas. El primero
verifica que el número de Péclet de celda no supere el valor dos, el
segundo que la masa cierre el balance con el término de decaimiento y el
tercero que el orden observado bajo refinamiento coincida con el teórico.
Aquí se aplican los tres al vertimiento del Ejemplo 3.4.

In [ ]:
from scipy.sparse import diags, identity

RIO = dict(u=0.35, D=12.0, k=0.25 / 86400, M=25.0, area=18.0, x0=1200.0)
LARGO, T_INICIAL, T_FINAL = 5000.0, 600.0, 3600.0


def perfil_exacto(x, t, u, D, k, M, area, x0):
    """Solución analítica de una inyección instantánea, en kg/m3."""
    s2 = 4 * D * t
    return (M / (area * np.sqrt(np.pi * s2))
            * np.exp(-(x - x0 - u * t)**2 / s2) * np.exp(-k * t))


def matriz_transporte(n, dx, u, D, k):
    """Diferencias centradas para advección, difusión y decaimiento."""
    sub = np.full(n - 1, D / dx**2 + u / (2 * dx))
    dia = np.full(n, -2 * D / dx**2 - k)
    sup = np.full(n - 1, D / dx**2 - u / (2 * dx))
    return diags([sub, dia, sup], [-1, 0, 1], format="csc")


def metodo_de_lineas(n_intervalos, rtol=1e-11, atol=1e-16):
    """Perfil en el instante final para una malla de n intervalos."""
    malla = np.linspace(0.0, LARGO, n_intervalos + 1)
    x_int = malla[1:-1]
    dx = malla[1] - malla[0]
    A = matriz_transporte(x_int.size, dx, RIO["u"], RIO["D"], RIO["k"])
    sol = solve_ivp(lambda t, c: A @ c, (T_INICIAL, T_FINAL),
                    perfil_exacto(x_int, T_INICIAL, **RIO), method="BDF",
                    jac=lambda t, c: A, rtol=rtol, atol=atol,
                    t_eval=[T_FINAL])
    return dx, x_int, sol.y[:, 0]

### Ejercicio 5

El orden observado es la pendiente de la recta que ajusta el logaritmo
del error máximo frente al logaritmo del paso espacial. Complete la
función y compruebe que vale dos, coherente con el error de truncamiento
de la aproximación centrada de la segunda derivada. Si la pendiente se
aplana en las mallas finas, algún error independiente del paso domina el
residuo, y las tres causas habituales son una frontera artificial
demasiado cercana, una tolerancia temporal insuficiente y un error en la
condición inicial.

In [ ]:
# COMPLETE: devuelva la pendiente del ajuste de log(error) contra log(dx).
REVISAR_ORDEN_ESPACIAL = False


def orden_observado(pasos, errores) -> float:
    """Orden de convergencia espacial estimado por mínimos cuadrados."""
    return 1.0            # marcador de posición

In [ ]:
mallas = (125, 250, 500, 1000, 2000)
pasos, errores = [], []
for n in mallas:
    dx, x_int, c_num = metodo_de_lineas(n)
    pasos.append(dx)
    errores.append(float(np.max(np.abs(c_num - perfil_exacto(x_int, T_FINAL, **RIO)))))

refinamiento = pd.DataFrame({"dx (m)": pasos, "error máximo (kg/m3)": errores})
refinamiento["razón"] = [np.nan] + list(np.array(errores[:-1]) / np.array(errores[1:]))
print(refinamiento.to_string(index=False,
                             formatters={"error máximo (kg/m3)": "{:.3e}".format}))

p_obs = orden_observado(pasos, errores)
comparar("orden observado bajo refinamiento", p_obs,
         LIBRO["rio_orden_observado"], 0.02, "")
comparar("error máximo con dx de 20 m", errores[1],
         LIBRO["rio_error_maximo"], 5e-8, "kg/m3")

if REVISAR_ORDEN_ESPACIAL:
    assert abs(p_obs - 2.0) < 0.02, "el orden espacial debe valer dos"
    print("el error se divide por cuatro cada vez que la malla se refina por dos")
else:
    print("complete la celda anterior y ponga REVISAR_ORDEN_ESPACIAL = True")

### Ejercicio 6

El número de Courant \(C = u\,\Delta t/\Delta x\) y el número de
difusión \(d = D\,\Delta t/\Delta x^{2}\) resumen las dos restricciones
del esquema explícito. El análisis de von Neumann añade \(d \le 1/2\),
casi siempre más severa que la de Courant en problemas dominados por la
dispersión. Complete los dos pasos máximos y compruebe que con
\(\Delta x = 20\) m la difusión limita el paso a 16.67 s y la advección
a 57.14 s.

In [ ]:
# COMPLETE: paso máximo por difusión, dx**2/(2*D)
#           paso máximo por advección, dx/u
REVISAR_CFL = False


def pasos_maximos(dx: float, u: float, D: float) -> dict:
    """Paso de tiempo máximo por difusión y por advección, en segundos."""
    return dict(difusion=0.0, adveccion=0.0)

In [ ]:
limites = pasos_maximos(20.0, RIO["u"], RIO["D"])
ok = [comparar("paso máximo por difusión", limites["difusion"],
               LIBRO["rio_paso_difusion"], 5e-3, "s"),
      comparar("paso máximo por advección", limites["adveccion"],
               LIBRO["rio_paso_adveccion"], 5e-3, "s")]

if REVISAR_CFL:
    assert all(ok), "los pasos máximos no coinciden con la sección 3.3.2"
    print("la difusión gobierna, tal como corresponde a un problema disperso")
else:
    print("complete la celda anterior y ponga REVISAR_CFL = True")

La verificación del Ejemplo 3.4 pone cifras a la advertencia. Con un paso
de 15 s, es decir un número de difusión de 0.45, el esquema explícito
reproduce el perfil con error de 6.3 × 10⁻⁵ kg/m³. Con 20 s, es decir
0.60, la solución diverge y alcanza valores del orden de
5 × 10² kg/m³. Una violación moderada no siempre se manifiesta de
inmediato, porque el modo inestable crece desde el nivel del error de
redondeo, razón por la cual la condición se verifica antes de correr.

In [ ]:
def marcha_explicita(dt: float, n_intervalos: int = 250):
    """Euler explícito sobre el sistema del método de líneas."""
    malla = np.linspace(0.0, LARGO, n_intervalos + 1)
    x_int = malla[1:-1]
    dx = malla[1] - malla[0]
    A = matriz_transporte(x_int.size, dx, RIO["u"], RIO["D"], RIO["k"])
    c = perfil_exacto(x_int, T_INICIAL, **RIO)
    for _ in range(int(round((T_FINAL - T_INICIAL) / dt))):
        c = c + dt * (A @ c)
    return x_int, c, RIO["D"] * dt / dx**2


filas = []
for dt in (15.0, 20.0):
    x_int, c_exp, d_num = marcha_explicita(dt)
    error = float(np.max(np.abs(c_exp - perfil_exacto(x_int, T_FINAL, **RIO))))
    filas.append((dt, d_num, error, float(np.max(np.abs(c_exp)))))
explicito = pd.DataFrame(filas, columns=["dt (s)", "número de difusión",
                                         "error máximo (kg/m3)",
                                         "máximo absoluto (kg/m3)"])
print(explicito.to_string(index=False,
                          formatters={"error máximo (kg/m3)": "{:.2e}".format,
                                      "máximo absoluto (kg/m3)": "{:.2e}".format}))
comparar("error con dt de 15 s", explicito.loc[0, "error máximo (kg/m3)"],
         LIBRO["rio_error_explicito_d045"], 5e-7, "kg/m3")
assert explicito.loc[1, "máximo absoluto (kg/m3)"] > 1e2, \
    "con d igual a 0.60 la solución explícita debe divergir"
print("con d igual a 0.60 la solución explícita diverge, como anuncia el libro")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4.0))
ax1.loglog(pasos, errores, "o-", color=COLORES["azul"], label="error observado")
ax1.loglog(pasos, errores[0] * (np.array(pasos) / pasos[0])**2, "--",
           color=COLORES["rojo"], lw=1.0, label="pendiente dos")
ax1.set_xlabel("paso espacial dx (m)")
ax1.set_ylabel("error máximo (kg/m3)")
ax1.set_title(f"Convergencia espacial, orden observado {p_obs:.2f}")
ax1.legend()

x_int, c15, _ = marcha_explicita(15.0)
x_int, c20, _ = marcha_explicita(20.0)
ax2.plot(x_int / 1000, 1e3 * perfil_exacto(x_int, T_FINAL, **RIO),
         color=COLORES["gris"], lw=1.2, label="analítica")
ax2.plot(x_int / 1000, 1e3 * c15, color=COLORES["verde"], lw=1.0,
         label="explícito, d = 0.45")
ax2.plot(x_int / 1000, 1e3 * np.clip(c20, -5, 5), color=COLORES["rojo"],
         lw=0.8, label="explícito, d = 0.60, recortado")
ax2.set_xlim(1.5, 3.5)
ax2.set_ylim(-3, 4)
ax2.set_xlabel("abscisa x (km)")
ax2.set_ylabel("concentración c (mg/L)")
ax2.set_title("Efecto de violar el límite de difusión")
ax2.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

## 6. Problemas del capítulo

El problema 3-10 enfría un lote de 500 kg de leche desde 35 hasta
4 grados Celsius contra un refrigerante a 1 grado, con \(UA = 399\) W/K
y calor específico de 3.93 kJ/(kg K), y pide la solución analítica. El
problema 3-11 repite el cálculo con tolerancias de 10⁻³, 10⁻⁶ y 10⁻⁹ y
pide discutir si el error respeta la tolerancia. El problema 3-12 tiene
autovalores de menos 4500 y menos 0.02 por segundo y pide la razón de
rigidez, el paso máximo de Euler explícito y los pasos para simular
600 s. Los tres se resuelven con las funciones ya escritas.

In [ ]:
MASA, CP, UA, T_REFRIG = 500.0, 3930.0, 399.0, 1.0     # kg, J/(kg K), W/K, C
TAU_LECHE = MASA * CP / UA                              # s
T_ANALITICO = TAU_LECHE * np.log((35.0 - T_REFRIG) / (4.0 - T_REFRIG))

enfriamiento = lambda t, T: [-(T[0] - T_REFRIG) / TAU_LECHE]


def umbral_cuatro(t, T):
    return T[0] - 4.0


umbral_cuatro.direction = -1
umbral_cuatro.terminal = True

print(f"problema 3-10  constante de tiempo {TAU_LECHE / 3600:.3f} h")
print(f"               tiempo analítico    {T_ANALITICO / 3600:.4f} h")

filas = []
for rtol in (1e-3, 1e-6, 1e-9):
    sol = solve_ivp(enfriamiento, (0.0, 6 * TAU_LECHE), [35.0], method="RK45",
                    rtol=rtol, atol=rtol * 1e-3, dense_output=True,
                    events=umbral_cuatro)
    t_num = float(sol.t_events[0][0])
    filas.append((rtol, t_num / 3600, sol.nfev,
                  abs(t_num - T_ANALITICO) / T_ANALITICO))
print()
print(pd.DataFrame(filas, columns=["rtol", "tiempo (h)", "evaluaciones",
                                   "error relativo"]).to_string(
    index=False, formatters={"rtol": "{:.0e}".format,
                             "error relativo": "{:.2e}".format}))

lam = np.array([-4500.0, -0.02])
print(f"\nproblema 3-12  razón de rigidez {razon_rigidez(lam):,.0f}")
print(f"               paso máximo de Euler {paso_maximo_euler(lam) * 1e3:.3f} ms")
print(f"               pasos para 600 s {600 / paso_maximo_euler(lam):,.0f}")

## 7. Cierre

Al terminar este cuaderno el estudiante debe poder hacer lo siguiente.

1. Correr un mismo modelo bajo varias condiciones de operación y
   reconocer qué parte de la respuesta cambia y qué parte no puede
   cambiar por una relación de conservación. Revise el Ejemplo 3.2 si el
   invariante se degrada.
2. Medir el error global frente a una referencia independiente y explicar
   por qué puede exceder la tolerancia solicitada. Revise la Ecuación 3.9
   y el comentario que la sigue.
3. Calcular la razón de rigidez a partir de los autovalores del jacobiano
   y anticipar el paso que la estabilidad permite. Revise la Definición
   3.3 y la Figura 3.6.
4. Diagnosticar la rigidez comparando conteos de evaluaciones y elegir el
   integrador con la Tabla 3.2 en la mano.
5. Verificar el orden espacial de un método de líneas y comprobar el
   límite de estabilidad de su versión explícita antes de correr, no
   después. Revise el Teorema 3.2 y el Algoritmo 3.2.

Con el modelo bajo control, el paso siguiente consiste en construir
escenarios y comparar alternativas, que es el asunto del cuaderno tercero.